## In-Class Lecture 39-1 - Qualify an Engine

A spacecraft constantly rotates vectors between coordinate frames: where the Sun is relative to the solar panels, where Earth is relative to the antenna. In this final lecture, you will write that rotation for a DSP, in 16-bit fixed point, choose it at build time, and put it through the same qualification pipeline from Reading 39-2. Time permitting, you will do the same for an NPU.

### Reviewing Opening your Environment on the Notre Dame Machines

<b>1.</b> Log into the Notre Dame Machines as you performed on Homework 1 using Tabby Terminal 

Open Tabby Terminal on your machine.

Type in: <code>ssh -Y username@student13.cse.nd.edu</code>
<ul><li>"username" is your NetID.</li>
<ul>
<li>Example: my email is matt.morrison@nd.edu, but my NetID is mmorri22, so I would enter:<p>
<code>ssh -Y mmorri22@student13.cse.nd.edu</code><p>
The <code>-Y</code> is included to treat the Notre Dame machines as a trusted client<p>
</li>
</ul>
</ul>

2. Next, we will create a directory where you will save your In-Class Code. First, we will go into the Dropbox directory where you will submit your code:

<code>cd esc-courses</code>

<code>cd fa26-cse-20133.01</code>

<code>cd dropbox</code>

Make a directory for you to practice your code for this lecture

    mkdir inclass39
    cd inclass39

Enable script so you can run the most up-to-date versions of C and C++:

    scl enable gcc-toolset-15 bash

> <b>Note:</b> If you use the <code>-std=c2x</code> and get the following error, this means you did not run the enable command<br>
> <code>gcc: error: unrecognized command line option ‘-std=c2x’; did you mean ‘-std=c1x’?</code>

To obtain the code for today's lecture, perform the following commands:

    wget https://raw.githubusercontent.com/mmorri22/cse20133/main/inclass/lec39/setup.sh
    chmod a+rx setup.sh
    ./setup.sh
    
To confirm, perform the <code>ls</code> command. You will see the following:

    engine_fp32.c  engine.h  engine_int8.c  engine_q15.c  Makefile  qualify_engine.c  reference.c

### In-Class Reading Discussion

<font color = "red">Class Introduction Question #1 - What is a heterogeneous computer? Using Ama's pattern, which engine on a spacecraft should run the safety loop, and why?</font>

<font color = "red">Class Introduction Question #2 - <code>precision.c</code> computed the same product four ways and got four answers. Why, and how do you decide whether an engine's answer is acceptable?</font>

<font color = "red">Class Introduction Question #3 - When the Q15 sum overflowed, why is the saturated result (0.99997) safer than the wrapped result (0.95947), even though both are wrong?</font>

<font color = "red">Class Introduction Question #4 - What is continuous qualification? In the <code>qualify</code> pipeline, which stages can a tool decide by itself, and which stage still needs a person?</font>

<font color = "red">Class Introduction Question #5 - Why do flight software teams test on an emulator like QEMU, and what can an emulator not tell them?</font>

<font color = "red">Class Introduction Question #6 - Pick one of Ama's six takeaways. Describe the specific code you wrote this semester that put it into practice, and the failure it prevents.</font>

### The Files

<ul>
    <li><code>engine.h</code> - the interface every engine provides: a name, a <b>stated tolerance</b>, and a <code>rotate</code> function. Read it first.</li>
    <li><code>reference.c</code> - the FP64 reference. Every engine is judged against it.</li>
    <li><code>engine_fp32.c</code> - a complete FP32 engine. Use it as your model.</li>
    <li><code>engine_q15.c</code> and <code>engine_int8.c</code> - your engines.</li>
    <li><code>qualify_engine.c</code> - the qualification harness. It runs edge cases and 1000 random rotations, and checks two things: the error against the reference, and a law of physics. <b>A rotation never changes a vector's length.</b></li>
</ul>

Look at the top of the <code>Makefile</code>:

    ENGINE := fp32
    SOURCES := qualify_engine.c reference.c engine_$(ENGINE).c

Exactly one engine is linked, chosen when you build. This is the build-time selection from Reading 35-2: different hardware, a different file, and no function pointers.

### Step 0 - Qualify the Model Engine

    make test

The FP32 engine is qualified, with a worst error near 10<sup>-7</sup>. Now try the empty Q15 engine:

    make test ENGINE=q15

It is not qualified. Notice that you changed engines without editing any code.

### Step 1 - <code>to_q15</code>

In <code>engine_q15.c</code>, convert a <code>double</code> to Q15, as in <code>precision.c</code>:
<ol>
    <li>Multiply by 32768 and round to the nearest whole number with <code>round</code>.</li>
    <li>If the result is above 32767, return <code>INT16_MAX</code>. If it is below -32768, return <code>INT16_MIN</code>. <b>Saturate; never wrap.</b></li>
    <li>Otherwise, return it as an <code>int16_t</code>.</li>
</ol>

Rotation matrices and unit vectors contain only values between -1 and 1, so Q15 fits them with no extra scaling. Almost: which value between -1 and 1 does Q15 still <i>not</i> fit? The harness tests it.

### Step 2 - <code>rotate</code>

For each row <code>i</code>:
<ol>
    <li>Multiply each <code>to_q15(R[i][j])</code> by <code>to_q15(v[j])</code>, as 32-bit integers, and add the products into an <code>int64_t</code> accumulator.</li>
    <li>Each product has 30 fraction bits. Shift the sum right by 15 to get back to Q15.</li>
    <li>Saturate the result to the <code>int16_t</code> range.</li>
    <li>Convert back: <code>out[i]</code> is the Q15 value divided by 32768.0.</li>
</ol>

    make test ENGINE=q15

### Step 3 - The Full Pipeline

Run every check from this unit on your engine:

    make qualify ENGINE=q15

The pipeline builds with every warning as an error, runs the static analyzer, runs under AddressSanitizer and UBSan, measures accuracy against the reference, and finishes with Cobra. Your engine must reach <code>QUALIFICATION EVIDENCE COLLECTED</code>.

Cobra will report findings, including <code>Limit use of global data</code> for <code>ENGINE_NAME</code> and <code>ENGINE_TOLERANCE</code>. Should you fix those, or are they a deviation? Be ready to argue your answer.

### Step 4 - Time Permitting: The INT8 Engine

In <code>engine_int8.c</code>, write the NPU-style engine from <code>precision.c</code>: one scale factor for the matrix and one for the vector, values rounded to the range -127 to 127, products accumulated in a 32-bit integer, then scaled back.

<b>Before you run it, state your tolerance.</b> Set <code>ENGINE_TOLERANCE</code> and write a comment explaining your choice. With 8 bits, what is the smallest step between two values near 1? About how many steps of error could three rounded products add up to? Then:

    make test ENGINE=int8

If your engine fails, do not simply raise the tolerance until it passes. Either your code is wrong, or your reasoning was. Find out which.

Finally, qualify every engine in one command:

    make qualify_all

### Discussion

<ol>
    <li>Suppose <code>to_q15</code> wrapped instead of saturating. Rotating the x-axis by the identity would give -1.0 instead of 1.0. Would the <b>length</b> check catch that? Would the reference check? What does that tell you about relying on a single kind of test?</li>
    <li>All three engines are qualified. Which would you use to point a high-gain antenna at Earth, and which to classify craters in a camera image? Why?</li>
    <li>In Step 4, you stated a tolerance before measuring. Why does it matter that the tolerance comes first? What goes wrong if engineers choose the tolerance after seeing the results?</li>
</ol>

## Compilation Run

    make qualify ENGINE=q15
    make qualify_all

When you are finished, the Q15 engine reaches <code>QUALIFICATION EVIDENCE COLLECTED</code>. Time permitting, so does the INT8 engine, with a tolerance you stated and justified before running it.